# Emotion Classification using Natural Language Processing

This notebook demonstrates a complete NLP pipeline for emotion classification from text data. The project includes:

- **Data Loading & Exploration**: Loading emotion-labeled text data
- **Data Preprocessing**: Text cleaning, normalization, and feature preparation
- **Text Processing**: Tokenization, stopword removal, and text transformation
- **Machine Learning Pipeline**: Building and training emotion classification models

## Dataset Overview
The dataset contains text samples with corresponding emotion labels (sadness, anger, love, etc.)

---

## 1. Environment Setup & Dependencies

Installing required packages for data manipulation, visualization, and NLP processing.

In [29]:
# Install required packages for NLP processing and data analysis
# pandas: Data manipulation and analysis
# matplotlib & seaborn: Data visualization
# scikit-learn: Machine learning algorithms
# nltk: Natural Language Processing toolkit
%pip install pandas matplotlib seaborn scikit-learn nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Import Required Libraries

In [30]:
# Core libraries for data processing and visualization
import pandas as pd              # Data manipulation and analysis
import matplotlib.pyplot as plt  # Plotting and visualization
import seaborn as sns           # Statistical data visualization
import nltk                     # Natural Language Processing toolkit

## 2. Data Loading & Initial Exploration

Loading the emotion classification dataset and examining its structure.

In [31]:
# Load the emotion classification dataset
# File format: semicolon-separated values with text and emotion columns
df = pd.read_csv('train.txt', sep=';', header=None, names=["text", "emotion"])

# Display first 5 rows to understand data structure
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


## 3. Label Encoding

Converting categorical emotion labels to numerical values for machine learning algorithms.

In [32]:
# Create emotion-to-number mapping for machine learning compatibility
unique_emotion = df['emotion'].unique()  # Get all unique emotion labels
emotion_number = {}  # Dictionary to store emotion-number mappings

# Assign numerical values to each unique emotion
i = 0
for emotion in unique_emotion:
    emotion_number[emotion] = i
    i += 1

# Apply the mapping to convert emotions to numerical labels
df['emotion'] = df['emotion'].map(emotion_number)

# Display the transformed data
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


## 4. Text Preprocessing Pipeline

Comprehensive text cleaning and normalization to improve model performance.

### 4.1 Case Normalization

Converting all text to lowercase for consistency.

In [33]:
# Convert all text to lowercase for uniformity
# This ensures "Happy" and "happy" are treated as the same word
df['text'] = df['text'].apply(lambda x: x.lower())

### 4.2 Text Cleaning Functions

Removing punctuation, numbers, and non-ASCII characters that may interfere with analysis.

In [34]:
import string

def remove_punctuation(text):
    """
    Remove all punctuation marks from text.
    
    Args:
        text (str): Input text string
    
    Returns:
        str: Text without punctuation
    """
    return text.translate(str.maketrans('', '', string.punctuation))

def remove_numbers(text):
    """
    Remove all numeric digits from text.
    
    Args:
        text (str): Input text string
    
    Returns:
        str: Text without numbers
    """
    return text.translate(str.maketrans('', '', string.digits))

def remove_emojis(text):
    """
    Remove emojis and non-ASCII characters from text.
    
    Args:
        text (str): Input text string
    
    Returns:
        str: Text with only ASCII characters
    """
    return text.encode('ascii', 'ignore').decode('ascii')

# Apply all cleaning functions to the text data
df['text'] = df['text'].apply(remove_punctuation)  # Remove punctuation
df['text'] = df['text'].apply(remove_numbers)      # Remove numbers
df['text'] = df['text'].apply(remove_emojis)       # Remove emojis/non-ASCII

### 4.3 Stop Words Removal

Removing common English words (like "the", "and", "is") that don't contribute to emotion classification.

In [35]:
import nltk

# Download required NLTK data for tokenization and stopwords
nltk.download('punkt_tab')  # Tokenizer models
nltk.download('punkt')      # Sentence tokenizer
nltk.download('stopwords')  # English stopwords list

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lalit\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lalit\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lalit\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [36]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Load English stopwords (common words like 'the', 'and', 'is', etc.)
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    """
    Remove English stopwords from text while preserving meaningful words.
    
    Args:
        text (str): Input text string
    
    Returns:
        str: Text with stopwords removed
    """
    # Tokenize text into individual words
    tokens = word_tokenize(text)
    
    # Filter out stopwords (case-insensitive comparison)
    filtered_tokens = [word for word in tokens if word.lower() not in stop_words]
    
    # Rejoin filtered tokens into a single string
    return ' '.join(filtered_tokens)

# Apply stopword removal to all text data
df['text'] = df['text'].apply(remove_stopwords)

# Display the preprocessed data
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [37]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer

In [38]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.2, random_state=42)

In [39]:
bow_vectorizer = CountVectorizer()
tf_idf_vectorizer = TfidfVectorizer()

In [40]:
bow_X_train = bow_vectorizer.fit_transform(X_train)
bow_X_test = bow_vectorizer.transform(X_test)

tf_idf_X_train = tf_idf_vectorizer.fit_transform(X_train)
tf_idf_X_test = tf_idf_vectorizer.transform(X_test)

In [41]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import  accuracy_score

In [42]:
bow_naive_bayes_model = MultinomialNB()
bow_naive_bayes_model.fit(bow_X_train, y_train)

tf_idf_naive_bayes_model = MultinomialNB()
tf_idf_naive_bayes_model.fit(tf_idf_X_train, y_train)

bow_svc_model = SVC()
bow_svc_model.fit(bow_X_train, y_train)

tf_idf_svc_model = SVC()
tf_idf_svc_model.fit(tf_idf_X_train, y_train)

bow_logistic_model = LogisticRegression(max_iter=1000)
bow_logistic_model.fit(bow_X_train, y_train)

tf_idf_logistic_model = LogisticRegression(max_iter=1000)
tf_idf_logistic_model.fit(tf_idf_X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [43]:
bow_svc_model_pred_nb = bow_naive_bayes_model.predict(bow_X_test)
bow_svc_model_pred_svc = bow_svc_model.predict(bow_X_test)
bow_logistic_model_pred_log = bow_logistic_model.predict(bow_X_test)



print("Naive Bayes Accuracy", accuracy_score(y_pred=bow_svc_model_pred_nb, y_true=y_test))
print("SVC Accuracy", accuracy_score(y_pred=bow_svc_model_pred_svc, y_true=y_test))
print("Logistic Regression Accuracy", accuracy_score(y_pred=bow_logistic_model_pred_log, y_true=y_test))

Naive Bayes Accuracy 0.7678125
SVC Accuracy 0.8225
Logistic Regression Accuracy 0.88875


In [44]:
tf_idf_svc_model_pred_nb = tf_idf_naive_bayes_model.predict(tf_idf_X_test)
tf_idf_svc_model_pred_svc = tf_idf_svc_model.predict(tf_idf_X_test)
tf_idf_logistic_model_pred_log = tf_idf_logistic_model.predict(tf_idf_X_test)

print("Naive Bayes Accuracy", accuracy_score(y_pred=tf_idf_svc_model_pred_nb, y_true=y_test))
print("SVC Accuracy", accuracy_score(y_pred=tf_idf_svc_model_pred_svc, y_true=y_test))
print("Logistic Regression Accuracy", accuracy_score(y_pred=tf_idf_logistic_model_pred_log, y_true=y_test))

Naive Bayes Accuracy 0.6609375
SVC Accuracy 0.8515625
Logistic Regression Accuracy 0.8615625
